In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("../data/processed/image_metadata.csv")

print(df.shape)
df.head()

(4139, 6)


,room,style,path,width,height,aspect_ratio
0,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,421,0.560570
1,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,472,0.500000
2,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,236,1.000000
3,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...,183,275,0.665455
4,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...,251,201,1.248756


In [2]:
print("Rooms:")
print(df["room"].value_counts())
print("\nStyles:")
print(df["style"].value_counts())

Rooms:
room
kitchen        1498
living_room    1056
bedroom         881
bathroom        704
Name: count, dtype: int64

Styles:
style
modern          1026
boho             918
industrial       859
minimalist       744
scandinavian     592
Name: count, dtype: int64


In [3]:
evaluation_queries = (
    df.groupby(["room","style"], group_keys = False).apply(
        lambda x: x.sample(
            min(len(x),2),
            random_state =42
        )
    ). reset_index(drop = True) 
)

In [4]:
print("Number of candidate queries:", len(evaluation_queries))

Number of candidate queries: 40


In [6]:
evaluation_queries = (
    df.groupby(["room", "style"], group_keys=False)
      .sample(n=2, random_state=42)
      .reset_index(drop=True)
)

In [8]:
evaluation_queries[
    ["room","style","path"]
].head(10)

,room,style,path
0,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...
1,bathroom,boho,C:\Users\Khushi Trivedi\visual-retrieval-syste...
2,bathroom,industrial,C:\Users\Khushi Trivedi\visual-retrieval-syste...
3,bathroom,industrial,C:\Users\Khushi Trivedi\visual-retrieval-syste...
4,bathroom,minimalist,C:\Users\Khushi Trivedi\visual-retrieval-syste...
5,bathroom,minimalist,C:\Users\Khushi Trivedi\visual-retrieval-syste...
6,bathroom,modern,C:\Users\Khushi Trivedi\visual-retrieval-syste...
7,bathroom,modern,C:\Users\Khushi Trivedi\visual-retrieval-syste...
8,bathroom,scandinavian,C:\Users\Khushi Trivedi\visual-retrieval-syste...
9,bathroom,scandinavian,C:\Users\Khushi Trivedi\visual-retrieval-syste...


In [9]:
print("Number of candidate queries:", len(evaluation_queries))

Number of candidate queries: 40


In [10]:
evaluation_queries = (
    evaluation_queries
    .sample(
        n=min(30, len(evaluation_queries)),
        random_state=42
    )
    .reset_index(drop=True)
)

In [11]:
print("Final number of queries:", len(evaluation_queries))

Final number of queries: 30


In [12]:
evaluation_queries.insert(
    0,
    "query_id",
    range(len(evaluation_queries))
)

In [13]:
evaluation_queries.head()

,query_id,room,style,path,width,height,aspect_ratio
0,0,bedroom,scandinavian,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,419,0.563246
1,1,bedroom,modern,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,354,0.666667
2,2,bedroom,minimalist,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,420,0.561905
3,3,kitchen,modern,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,292,0.808219
4,4,bathroom,minimalist,C:\Users\Khushi Trivedi\visual-retrieval-syste...,236,354,0.666667


In [14]:
evaluation_queries["room"].value_counts()

room
bathroom       9
living_room    8
bedroom        7
kitchen        6
Name: count, dtype: int64

In [15]:
evaluation_queries.groupby(
    ["room", "style"]
).size()

room         style       
bathroom     boho            2
             industrial      2
             minimalist      2
             modern          1
             scandinavian    2
bedroom      boho            1
             industrial      2
             minimalist      1
             modern          2
             scandinavian    1
kitchen      boho            1
             minimalist      2
             modern          2
             scandinavian    1
living_room  boho            2
             industrial      1
             minimalist      2
             modern          2
             scandinavian    1
dtype: int64

In [16]:
os.makedirs(
    "../data/evaluation",
    exist_ok=True
)

In [17]:
evaluation_queries.to_csv(
    "../data/evaluation/evaluation_queries.csv",
    index=False
)

In [18]:
print(
    "Saved",
    len(evaluation_queries),
    "evaluation queries"
)

Saved 30 evaluation queries


In [20]:
import torch 

In [21]:
clip_embeddings = torch.load(
    "../data/embeddings/clip_embeddings.pt"
)

resnet_embeddings = torch.load(
    "../data/embeddings/resnet_embeddings.pt"
)

color_embeddings = torch.load(
    "../data/embeddings/color_histograms.pt"
)

In [22]:
def retrieve_top_k(
    query_embedding,
    stored_embeddings,
    df,
    query_path,
    top_k=5
):
    similarities = stored_embeddings @ query_embedding

    # Find the query image in the dataset
    matches = df.index[
        df["path"] == query_path
    ].tolist()

    # Don't retrieve the query itself
    if matches:
        similarities[matches[0]] = -1

    top_scores, top_indices = torch.topk(
        similarities,
        k=top_k
    )

    return top_indices, top_scores

In [23]:
all_results = []

for _, query in evaluation_queries.iterrows():

    query_id = query["query_id"]
    query_path = query["path"]

    # =========================
    # CLIP
    # =========================

    query_index = df.index[
        df["path"] == query_path
    ][0]

    query_embedding = clip_embeddings[query_index]

    indices, scores = retrieve_top_k(
        query_embedding,
        clip_embeddings,
        df,
        query_path,
        top_k=5
    )

    for rank, (idx, score) in enumerate(
        zip(indices, scores),
        start=1
    ):
        candidate = df.iloc[idx.item()]

        all_results.append({
            "query_id": query_id,
            "method": "CLIP",
            "rank": rank,
            "candidate_index": idx.item(),
            "candidate_path": candidate["path"],
            "candidate_room": candidate["room"],
            "candidate_style": candidate["style"],
            "similarity": score.item()
        })


    # =========================
    # ResNet
    # =========================

    query_embedding = resnet_embeddings[query_index]

    indices, scores = retrieve_top_k(
        query_embedding,
        resnet_embeddings,
        df,
        query_path,
        top_k=5
    )

    for rank, (idx, score) in enumerate(
        zip(indices, scores),
        start=1
    ):
        candidate = df.iloc[idx.item()]

        all_results.append({
            "query_id": query_id,
            "method": "ResNet",
            "rank": rank,
            "candidate_index": idx.item(),
            "candidate_path": candidate["path"],
            "candidate_room": candidate["room"],
            "candidate_style": candidate["style"],
            "similarity": score.item()
        })


    # =========================
    # Color Histogram
    # =========================

    query_embedding = color_embeddings[query_index]

    indices, scores = retrieve_top_k(
        query_embedding,
        color_embeddings,
        df,
        query_path,
        top_k=5
    )

    for rank, (idx, score) in enumerate(
        zip(indices, scores),
        start=1
    ):
        candidate = df.iloc[idx.item()]

        all_results.append({
            "query_id": query_id,
            "method": "Color Histogram",
            "rank": rank,
            "candidate_index": idx.item(),
            "candidate_path": candidate["path"],
            "candidate_room": candidate["room"],
            "candidate_style": candidate["style"],
            "similarity": score.item()
        })

In [24]:
candidate_results = pd.DataFrame(
    all_results
)

print(
    "Shape:",
    candidate_results.shape
)

Shape: (450, 8)


In [25]:
candidate_results.head(15)

,query_id,method,rank,candidate_index,candidate_path,candidate_room,candidate_style,similarity
0,0,CLIP,1,1175,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,minimalist,0.898707
1,0,CLIP,2,1542,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,scandinavian,0.898707
2,0,CLIP,3,1032,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,industrial,0.895409
3,0,CLIP,4,1514,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,scandinavian,0.887382
4,0,CLIP,5,781,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,boho,0.886600
5,0,ResNet,1,3166,C:\Users\Khushi Trivedi\visual-retrieval-syste...,living_room,boho,0.771968
6,0,ResNet,2,1481,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,scandinavian,0.766424
7,0,ResNet,3,944,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,industrial,0.766424
8,0,ResNet,4,1549,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,scandinavian,0.765797
9,0,ResNet,5,783,C:\Users\Khushi Trivedi\visual-retrieval-syste...,bedroom,boho,0.765796


In [26]:
print(
    candidate_results.groupby(
        "method"
    ).size()
)

method
CLIP               150
Color Histogram    150
ResNet             150
dtype: int64


In [27]:
print(
    candidate_results.groupby(
        ["query_id", "method"]
    ).size().value_counts()
)

5    90
Name: count, dtype: int64


In [28]:
candidate_results.to_csv(
    "../data/evaluation/candidate_results.csv",
    index=False
)

In [29]:
print(
    "Saved",
    len(candidate_results),
    "candidate results"
)

Saved 450 candidate results
